In [2]:
# Uncomment if needed

!pip install pandas numpy networkx matplotlib seaborn
!pip install spacy
!pip install yfinance
!pip install requests beautifulsoup4
!pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 2.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [seaborn]m1/2 [seaborn]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 3.1 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [yfinance]5/6 [yfinance]]]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 3.6 MB/s  0:00:01 eta 0:00:01


In [3]:
import pandas as pd
import numpy as np

import requests
from bs4 import BeautifulSoup

import networkx as nx

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

import os

plt.style.use("ggplot")

In [4]:
folders = [
    "data",
    "data/raw",
    "data/processed",
    "graphs",
    "models"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created.")

Folders created.


In [5]:
import requests
from bs4 import BeautifulSoup

url = "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm"

html = requests.get(url)

print(html.status_code)

200


In [6]:
soup = BeautifulSoup(html.text, "html.parser")

links = []

for a in soup.find_all("a", href=True):

    href = a["href"]

    if "minutes" in href.lower():

        if href.endswith(".htm") or href.endswith(".pdf"):

            links.append(href)

links[:20]

['/monetarypolicy/files/fomcminutes20260128.pdf',
 '/monetarypolicy/fomcminutes20260128.htm',
 '/monetarypolicy/files/fomcminutes20260318.pdf',
 '/monetarypolicy/fomcminutes20260318.htm',
 '/monetarypolicy/files/fomcminutes20260429.pdf',
 '/monetarypolicy/fomcminutes20260429.htm',
 '/monetarypolicy/files/fomcminutes20250129.pdf',
 '/monetarypolicy/fomcminutes20250129.htm',
 '/monetarypolicy/files/fomcminutes20250319.pdf',
 '/monetarypolicy/fomcminutes20250319.htm',
 '/monetarypolicy/files/fomcminutes20250507.pdf',
 '/monetarypolicy/fomcminutes20250507.htm',
 '/monetarypolicy/files/fomcminutes20250618.pdf',
 '/monetarypolicy/fomcminutes20250618.htm',
 '/monetarypolicy/files/fomcminutes20250730.pdf',
 '/monetarypolicy/fomcminutes20250730.htm',
 '/monetarypolicy/files/fomcminutes20250917.pdf',
 '/monetarypolicy/fomcminutes20250917.htm',
 '/monetarypolicy/files/fomcminutes20251029.pdf',
 '/monetarypolicy/fomcminutes20251029.htm']

In [7]:
base = "https://www.federalreserve.gov"

minutes = []

for link in links:

    full = base + link

    print(full)

    page = requests.get(full)

    soup = BeautifulSoup(page.text, "html.parser")

    text = soup.get_text(separator=" ")

    minutes.append({
        "url": full,
        "text": text
    })

https://www.federalreserve.gov/monetarypolicy/files/fomcminutes20260128.pdf
https://www.federalreserve.gov/monetarypolicy/fomcminutes20260128.htm
https://www.federalreserve.gov/monetarypolicy/files/fomcminutes20260318.pdf
https://www.federalreserve.gov/monetarypolicy/fomcminutes20260318.htm
https://www.federalreserve.gov/monetarypolicy/files/fomcminutes20260429.pdf
https://www.federalreserve.gov/monetarypolicy/fomcminutes20260429.htm
https://www.federalreserve.gov/monetarypolicy/files/fomcminutes20250129.pdf
https://www.federalreserve.gov/monetarypolicy/fomcminutes20250129.htm
https://www.federalreserve.gov/monetarypolicy/files/fomcminutes20250319.pdf
https://www.federalreserve.gov/monetarypolicy/fomcminutes20250319.htm
https://www.federalreserve.gov/monetarypolicy/files/fomcminutes20250507.pdf
https://www.federalreserve.gov/monetarypolicy/fomcminutes20250507.htm
https://www.federalreserve.gov/monetarypolicy/files/fomcminutes20250618.pdf
https://www.federalreserve.gov/monetarypolicy/fo

In [8]:
df = pd.DataFrame(minutes)

df.to_csv(
    "data/raw/fomc_minutes.csv",
    index=False
)

df.head()

,url,text
0,https://www.federalreserve.gov/monetarypolicy/...,%PDF-1.7\r%����\r\n105 0 obj\r< >\rendobj\r ...
1,https://www.federalreserve.gov/monetarypolicy/...,ï»¿ \n \n \n \n \n \n \n \n \n \n \n \n \n \n ...
2,https://www.federalreserve.gov/monetarypolicy/...,%PDF-1.6\r%����\r\n297 0 obj\r< >\rendobj\r ...
3,https://www.federalreserve.gov/monetarypolicy/...,ï»¿ \n \n \n \n \n \n \n \n \n \n \n \n \n \n ...
4,https://www.federalreserve.gov/monetarypolicy/...,%PDF-1.6\r%����\r\n447 0 obj\r< >stream\r\nhޜU...


In [9]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r"\n", " ", text)

    text = re.sub(r"\s+", " ", text)

    text = re.sub(r"[^a-zA-Z0-9 ]", "", text)

    return text

In [10]:
df["clean_text"] = df["text"].apply(clean_text)

df.head()

,url,text,clean_text
0,https://www.federalreserve.gov/monetarypolicy/...,%PDF-1.7\r%����\r\n105 0 obj\r< >\rendobj\r ...,pdf17 105 0 obj endobj 123 0 obj filterfla...
1,https://www.federalreserve.gov/monetarypolicy/...,ï»¿ \n \n \n \n \n \n \n \n \n \n \n \n \n \n ...,the fed monetary policy skip to main content...
2,https://www.federalreserve.gov/monetarypolicy/...,%PDF-1.6\r%����\r\n297 0 obj\r< >\rendobj\r ...,pdf16 297 0 obj endobj 314 0 obj filterfla...
3,https://www.federalreserve.gov/monetarypolicy/...,ï»¿ \n \n \n \n \n \n \n \n \n \n \n \n \n \n ...,the fed monetary policy skip to main content...
4,https://www.federalreserve.gov/monetarypolicy/...,%PDF-1.6\r%����\r\n447 0 obj\r< >stream\r\nhޜU...,pdf16 447 0 obj stream humo0ibhmciv lj4iaa1d...


In [11]:
df.to_csv(
    "data/processed/fomc_clean.csv",
    index=False
)